In [2]:
import pandas as pd

# Đọc dữ liệu từ file Excel
df = pd.read_excel('TRAIN2.xlsx')

# 1. Xem danh sách chính xác tên các cột
print("Danh sách các cột trong file là:", df.columns.tolist())

# 2. Xem 5 dòng đầu tiên của bảng dữ liệu
display(df.head()) # Lệnh display trong Jupyter sẽ hiển thị bảng rất đẹp, hoặc bạn có thể dùng print(df.head())


Danh sách các cột trong file là: ['final', 'midterm']


,final,midterm
0,2.56,0.70
1,2.78,0.97
2,6.35,5.43
3,5.99,4.99
4,7.67,7.09


In [5]:
# 1. Định nghĩa Mạng Neural đơn giản (chỉ có 1 lớp Linear, tương đương y = wx + b)
class SimpleLinearNet(nn.Module):
    def __init__(self):
        super(SimpleLinearNet, self).__init__()
        self.linear = nn.Linear(1, 1) # 1 input (midterm) -> 1 output (final)

    def forward(self, x):
        return self.linear(x)

# Khởi tạo mô hình, hàm mất mát và bộ tối ưu
simple_model = SimpleLinearNet()
criterion = nn.MSELoss()
optimizer = optim.SGD(simple_model.parameters(), lr=0.01)

# 2. Vòng lặp huấn luyện
epochs = 1000
for epoch in range(epochs):
    y_pred = simple_model(X_tensor)
    loss = criterion(y_pred, y_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# ==========================================
# 3. CELL NÀY ĐỂ IN RA w VÀ b SAU KHI TRAIN
# ==========================================
print("=== KẾT QUẢ HUẤN LUYỆN ===")
# Trong PyTorch, w chính là 'weight', b chính là 'bias' của lớp Linear
w = simple_model.linear.weight.item()
b = simple_model.linear.bias.item()

print(f"Trọng số w = {w:.4f}")
print(f"Sai số b   = {b:.4f}")
print(f"Phương trình học được: y_pred = {w:.4f} * Midterm + {b:.4f}")


=== KẾT QUẢ HUẤN LUYỆN ===
Trọng số w = 0.8027
Sai số b   = 1.9841
Phương trình học được: y_pred = 0.8027 * Midterm + 1.9841


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

# 1. Đọc và chuẩn bị dữ liệu (Chuyển thành PyTorch Tensors)
df = pd.read_excel('TRAIN2.xlsx')
# Giả lập X, y từ phần trước (bạn dùng df['...'].values như hướng dẫn cũ)
X_data = df['midterm'].values.astype(np.float32).reshape(-1, 1) # Định dạng lại thành ma trận 2D: [số_lượng_sample, 1]
y_data = df['final'].values.astype(np.float32).reshape(-1, 1)

# Chuyển Numpy array sang PyTorch Tensor
X_tensor = torch.from_numpy(X_data)
y_tensor = torch.from_numpy(y_data)

# 2. Xây dựng Đồ thị tính toán (Computational Graph / Neural Network)
class ScorePredictNet(nn.Module):
    def __init__(self):
        super(ScorePredictNet, self).__init__()
        # Lớp ẩn: 1 input -> 10 neurons ẩn
        self.layer1 = nn.Linear(1, 10) 
        self.relu = nn.ReLU()          # Hàm kích hoạt phi tuyến
        # Lớp đầu ra: 10 neurons ẩn -> 1 output
        self.layer2 = nn.Linear(10, 1) 

    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out

model = ScorePredictNet()

# 3. Định nghĩa Hàm Mất Mát và Bộ Tối Ưu
criterion = nn.MSELoss() # Hàm tính lỗi Mean Squared Error
optimizer = optim.SGD(model.parameters(), lr=0.01) # Khai báo learning_rate = 0.01

# 4. Vòng lặp Huấn luyện (Training Loop)
epochs = 1000
print("Bắt đầu huấn luyện mạng Neural Network...")

for epoch in range(epochs):
    # Bước 1: Lan truyền tiến (Forward pass) - Tính y_pred
    y_pred = model(X_tensor)
    
    # Bước 2: Tính sai số (Loss)
    loss = criterion(y_pred, y_tensor)
    
    # Bước 3: Lan truyền ngược (Backward pass) và Cập nhật trọng số
    optimizer.zero_grad() # Xóa đạo hàm cũ
    loss.backward()       # Tự động tính đạo hàm dựa trên Computational Graph
    optimizer.step()      # Cập nhật trọng số (w, b) cho toàn bộ các lớp
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# 5. Dự đoán thử nghiệm (Inference)
print("-" * 30)
model.eval() # Chuyển model sang chế độ dự đoán
with torch.no_grad():
    # Thử dự đoán nếu điểm giữa kỳ là 7.5
    test_score = torch.tensor([[7.5]], dtype=torch.float32)
    predicted_final = model(test_score)
    print(f"Điểm giữa kỳ: 7.5 -> Dự đoán điểm cuối kỳ: {predicted_final.item():.2f}")


Bắt đầu huấn luyện mạng Neural Network...
Epoch [100/1000], Loss: 0.0360
Epoch [200/1000], Loss: 0.0037
Epoch [300/1000], Loss: 0.0019
Epoch [400/1000], Loss: 0.0016
Epoch [500/1000], Loss: 0.0013
Epoch [600/1000], Loss: 0.0011
Epoch [700/1000], Loss: 0.0009
Epoch [800/1000], Loss: 0.0008
Epoch [900/1000], Loss: 0.0007
Epoch [1000/1000], Loss: 0.0006
------------------------------
Điểm giữa kỳ: 7.5 -> Dự đoán điểm cuối kỳ: 8.00


In [7]:
print("=== TRỌNG SỐ CỦA MẠNG NEURAL PHỨC TẠP ===")
# In ra toàn bộ w và b của tất cả các lớp trong mô hình
for name, param in model.named_parameters():
    print(f"Layer: {name}")
    print(param.data)
    print("-" * 30)


=== TRỌNG SỐ CỦA MẠNG NEURAL PHỨC TẠP ===
Layer: layer1.weight
tensor([[-0.7985],
        [-0.9807],
        [ 0.5138],
        [ 0.3217],
        [-0.0862],
        [ 0.0114],
        [-0.9753],
        [ 0.4391],
        [ 0.3573],
        [ 0.1178]])
------------------------------
Layer: layer1.bias
tensor([ 0.8517, -0.6664,  1.1034, -0.1847, -0.9901, -0.6042,  0.9215,  0.2379,
         0.8694,  0.1036])
------------------------------
Layer: layer2.weight
tensor([[-0.1116,  0.2328,  0.8597, -0.0625, -0.2026,  0.2376,  0.2814,  0.2844,
          0.7100,  0.0338]])
------------------------------
Layer: layer2.bias
tensor([0.3257])
------------------------------
